In [1]:
import json
import os
import pandas as pd
import matplotlib.pyplot as plt

import warnings

In [2]:
warnings.filterwarnings('ignore')

In [3]:
response_var = pd.read_parquet('../data/interim/storm_outages_2014_2023.parquet')

In [4]:
meteorological_path = '../data/raw/meteorological'
meteorological_files = os.listdir(meteorological_path)

In [30]:
def get_data_with_response_variable(met_file):
    try:
        #met_file = meteorological_files[0]
        episode_fips_id = met_file.split('.')[0]
        #  Read data
        path = os.path.join(meteorological_path, met_file)
        with open(path, 'r') as f:
            data = json.load(f)
        # Make a dataframe from information
        data = pd.DataFrame(data.get('properties').get('parameter'))
        data.index.rename('time', inplace=True)
        data.reset_index(inplace=True)
        data['episode_fips_id'] = episode_fips_id
        data['meteorological_current_datetime_val'] = pd.to_datetime(
            data['time'].apply(lambda x: x[0:4]) + 
            '-' + 
            data['time'].apply(lambda x: x[4:6]) +
            '-' + 
            data['time'].apply(lambda x: x[6:8]) +
            ' ' +
            data['time'].apply(lambda x: x[8:10]) +
            ':00:00'
        )
        # Modification to change the hour treshold.
        data['meteorological_nextHour_datetime_val'] = data['meteorological_current_datetime_val'] + pd.Timedelta(value=1, unit='hours')
        # Get the response var dataframe
        response_var_episode = response_var[response_var.episode_fips_id==episode_fips_id]
        meaning_dict = {
            'begin_datetime': 'storm_start', 
            'end_datetime': 'storm_end', 
            'run_start_time_min': 'outage_start', 
            'run_start_time_max': 'outage_end'
        }
        response_var_episode.rename(columns=meaning_dict, inplace=True)
        response_var_episode.drop_duplicates(['episode_fips_id'], keep='first', inplace=True)
        response_var_episode_columns = [
            'episode_fips_id', 
            'storm_start', 
            #'storm_end',
            'outage_start',
            'outage_end'
        ]
        # Join information
        data_rv = data.merge(
            response_var_episode[response_var_episode_columns],
            on='episode_fips_id',
            how='left'
        )
        data_rv['outage_start_rounded'] = data_rv.outage_start.dt.round('H')
        data_rv['outage_end_rounded'] = data_rv.outage_end.dt.round('H')
        data_rv = data_rv[data_rv.meteorological_nextHour_datetime_val <= data_rv.outage_start_rounded]
        data_rv = data_rv[data_rv.meteorological_nextHour_datetime_val >= data_rv.storm_start]
        data_rv.loc[data_rv.meteorological_nextHour_datetime_val == data_rv.outage_start_rounded, 'outage_in_an_hour'] = 1
        data_rv.loc[data_rv.meteorological_nextHour_datetime_val != data_rv.outage_start_rounded, 'outage_in_an_hour'] = 0
        data_rv['episode_fips_time_id'] = data_rv.episode_fips_id + '_' + data_rv.time.astype(str)
        data_rv.set_index('episode_fips_time_id', inplace=True)
        return data_rv
    except:
        return None


In [31]:
#data_test, response_test = get_data_with_response_variable('164119_06029.json')

In [32]:
response_test

,EPISODE_ID,fips_code_id,episode_description,storm_start,storm_end,storm_duration,episode_fips_id,storm_caused_outage,outage_index_id,outage_start_minus_storm_start,outage_end_minus_storm_end,outage_start_minus_storm_end,outage_end_minus_storm_start,outage_duration,outage_start,outage_end
288600,164119,06029,Although rainfall was above normal for Decembe...,2021-12-01,2021-12-31 23:59:00,743.983333,164119_06029,1.0,06029__0169,7.979167,-23.009722,-23.020139,7.989583,0.010417,2021-12-08 23:30:00,2021-12-08 23:45:00
288601,164119,06029,Although rainfall was above normal for Decembe...,2021-12-01,2021-12-31 23:59:00,743.983333,164119_06029,1.0,06029__0170,8.927083,-22.061806,-22.072222,8.937500,0.010417,2021-12-09 22:15:00,2021-12-09 22:30:00
288602,164119,06029,Although rainfall was above normal for Decembe...,2021-12-01,2021-12-31 23:59:00,743.983333,164119_06029,1.0,06029__0171,13.125000,-17.207639,-17.874306,13.791667,0.666667,2021-12-14 03:00:00,2021-12-14 19:00:00
288603,164119,06029,Although rainfall was above normal for Decembe...,2021-12-01,2021-12-31 23:59:00,743.983333,164119_06029,1.0,06029__0172,14.843750,-16.061806,-16.155556,14.937500,0.093750,2021-12-15 20:15:00,2021-12-15 22:30:00
288604,164119,06029,Although rainfall was above normal for Decembe...,2021-12-01,2021-12-31 23:59:00,743.983333,164119_06029,1.0,06029__0173,27.114583,-3.832639,-3.884722,27.166667,0.052083,2021-12-28 02:45:00,2021-12-28 04:00:00


In [33]:
data_rvs = []
for met_file in meteorological_files:
    data_rvs.append(get_data_with_response_variable(met_file))

In [34]:
cleaned_data_rvs = [x for x in data_rvs if x is not None]

In [35]:
all_data = pd.concat(cleaned_data_rvs)

In [36]:
all_data.outage_in_an_hour.sum()

np.float64(4882.0)

In [39]:
all_data.outage_in_an_hour.mean()

np.float64(0.021575516517511877)

In [38]:
all_data.drop(
    ['meteorological_nextHour_datetime_val', 
     'storm_start', 'outage_start', 'outage_end', 'outage_start_rounded', 'outage_end_rounded'], axis=1
).to_parquet('../data/interim/meteorological_data_with_outages.parquet')